# **MODELS**

In [1]:
from transformers import AutoTokenizer

e:\RAG-INTERN-PROJECT\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Document

In [2]:
from dataclasses import dataclass
"""
Represents a structured product documnet used in a RAG pipeline.
This documnet serves as the souce of truth for product information.
It stores product information before it is converted into text for chunking and embedding.
"""
@dataclass
class Document:
    document_id : int
    name : str
    category : str
    features : list[str]
    specifications: dict[str,str]
    description: str

    def to_text(self)->str:
        features_text = "\n".join(f"- {feature}" for feature in self.features)
        specifications_text = "\n".join(f"{key}: {value}" for key,value in self.specifications.items())
        return f"""Product Name: {self.name}
        Category: {self.category}
        Features: {features_text}
         Specifications: {specifications_text}
        Description: {self.description}"""
    def get_sections(self)->dict[str,str]:
        """
        Returns the doucumnet organized into semantic sections.
        This representaion is used for field-aware chunking.
        """
        features_text = "\n".join(f"- {feature}" for feature in self.features)
        specifications_text = "\n".join(f"{key}:{value}" for key,value in self.specifications.items())

        return {
            "identity":(
                f"Product Name:{self.name}\n"
                f"Category: {self.category}"
            ),
            "features": features_text,
            "specifications": specifications_text,
            "description": self.description
        }        

In [3]:
@dataclass
class Chunk:
    """
    Represnts a retreivable chunk generated form a product document.
    """
    chunk_id: int
    document_id: int
    text: str
    metadata: dict[str,str]
    chunk_index: int

## **Base Chunker**

In [4]:
from abc import ABC,abstractmethod

class BaseChunker(ABC):
    """
    Abstract Base Class for all chunking strategies.

    Every upcoming chunker implementaion must convert a
    Document into a list of chunk objects.

    """
    @abstractmethod
    def chunk(self,document: Document) -> list[Chunk]:
        """
        Split the doc. into retrievable chunks.
        Args: document - represnts the doc. to be chunked.
        Returns: A list of chunk objects.
        """
        pass

class FixedSizeChunker(BaseChunker):
    """
    splits a doc. with fixed size chunks with overlap.
    """
    def __init__(self,chunk_size:int, chunk_overlap: int = 0):
        if chunk_size <=0:
            raise ValueError("chunk_size must be greater than 0.")
        if chunk_overlap < 0:
            raise ValueError("chunk_overlap cannot be negative.")
        if chunk_overlap >=chunk_size:
            raise ValueError("chunk_overlap must be smaller than chunk_size.")
        self.chunk_size = chunk_size
        self.chunk_overlap = chunk_overlap

    def chunk(self,document:Document) -> list[Chunk]:
        text = document.to_text()
        chunks = []
        start = 0
        chunk_index = 0
        while start < len(text):
            end = min(start + self.chunk_size,len(text))
            chunk_text = text[start:end]
            chunk = Chunk(
                chunk_id = f"{document.document_id}_{chunk_index}",
                document_id = document.document_id,
                text = chunk_text,
                metadata = {"category": document.category},
                chunk_index = chunk_index
            )
            chunks.append(chunk)

            if end == len(text):
                break
            start = end -self.chunk_overlap
            chunk_index += 1
        return chunks

class FieldAwareChunker(BaseChunker):
    """
    Splits a structured product documnet into sematically meaningful chunks
    based on its fields.
    """
    def chunk(self,document: Document) ->list[Chunk]:
        sections = document.get_sections()
        chunks = []

        for index,(section_name,section_text) in enumerate(sections.items()):
            chunk = Chunk(
                chunk_id = f"{document.document_id}_{index}",
                document_id = document.document_id,
                text = section_text,
                metadata={
                    "category": document.category,
                    "section": section_name
                },
                chunk_index=index
            )
            chunks.append(chunk)
        return chunks


In [5]:
#TESTING

In [6]:
sample_doc = Document(
    document_id=1,
    name="Titan Gaming Laptop",
    category="Laptop",
    features=[
        "RTX 4080 GPU",
        "32GB DDR5 RAM",
        "1TB NVMe SSD",
        "WiFi 7"
    ],
    specifications={
        "Processor": "Intel Core Ultra 9",
        "Display": "16-inch QHD",
        "Battery": "90Wh",
        "Weight": "2.4 kg"
    },
    description=(
        "The Titan Gaming Laptop is designed for gamers and creators. "
        "It delivers excellent gaming performance, fast rendering speeds, "
        "and long battery life while maintaining an efficient cooling system."
    )
)


In [7]:
fixed_chunker = FixedSizeChunker(
    chunk_size=120,
    chunk_overlap=20
)
fixed_chunks = fixed_chunker.chunk(sample_doc)

In [8]:
print(f"Total Chunks: {len(fixed_chunks)}\n")

for chunk in fixed_chunks:
    print("=" * 60)
    print(f"Chunk ID      : {chunk.chunk_id}")
    print(f"Document ID   : {chunk.document_id}")
    print(f"Chunk Index   : {chunk.chunk_index}")
    print(f"Metadata      : {chunk.metadata}")
    print("\nChunk Text:")
    print(chunk.text)

Total Chunks: 5

Chunk ID      : 1_0
Document ID   : 1
Chunk Index   : 0
Metadata      : {'category': 'Laptop'}

Chunk Text:
Product Name: Titan Gaming Laptop
        Category: Laptop
        Features: - RTX 4080 GPU
- 32GB DDR5 RAM
- 1TB NVMe S
Chunk ID      : 1_1
Document ID   : 1
Chunk Index   : 1
Metadata      : {'category': 'Laptop'}

Chunk Text:
DR5 RAM
- 1TB NVMe SSD
- WiFi 7
         Specifications: Processor: Intel Core Ultra 9
Display: 16-inch QHD
Battery: 90W
Chunk ID      : 1_2
Document ID   : 1
Chunk Index   : 2
Metadata      : {'category': 'Laptop'}

Chunk Text:
nch QHD
Battery: 90Wh
Weight: 2.4 kg
        Description: The Titan Gaming Laptop is designed for gamers and creators. I
Chunk ID      : 1_3
Document ID   : 1
Chunk Index   : 3
Metadata      : {'category': 'Laptop'}

Chunk Text:
mers and creators. It delivers excellent gaming performance, fast rendering speeds, and long battery life while maintain
Chunk ID      : 1_4
Document ID   : 1
Chunk Index   : 4
Metadata   

In [9]:
field_chunker = FieldAwareChunker()

field_chunks = field_chunker.chunk(sample_doc)

In [10]:
print(f"Total Chunks: {len(field_chunks)}\n")

for chunk in field_chunks:
    print("=" * 60)
    print(f"Chunk ID      : {chunk.chunk_id}")
    print(f"Chunk Index   : {chunk.chunk_index}")
    print(f"Metadata      : {chunk.metadata}")
    print("\nChunk Text:")
    print(chunk.text)

Total Chunks: 4

Chunk ID      : 1_0
Chunk Index   : 0
Metadata      : {'category': 'Laptop', 'section': 'identity'}

Chunk Text:
Product Name:Titan Gaming Laptop
Category: Laptop
Chunk ID      : 1_1
Chunk Index   : 1
Metadata      : {'category': 'Laptop', 'section': 'features'}

Chunk Text:
- RTX 4080 GPU
- 32GB DDR5 RAM
- 1TB NVMe SSD
- WiFi 7
Chunk ID      : 1_2
Chunk Index   : 2
Metadata      : {'category': 'Laptop', 'section': 'specifications'}

Chunk Text:
Processor:Intel Core Ultra 9
Display:16-inch QHD
Battery:90Wh
Weight:2.4 kg
Chunk ID      : 1_3
Chunk Index   : 3
Metadata      : {'category': 'Laptop', 'section': 'description'}

Chunk Text:
The Titan Gaming Laptop is designed for gamers and creators. It delivers excellent gaming performance, fast rendering speeds, and long battery life while maintaining an efficient cooling system.


## Synthetic Dataset Generator

In [11]:
import random
class SyntheticDatasetGenerator:
    """
    Generate a synthetic corpus of realistic product documents for a RAG pipeline.
    The generator has differnt profiles, diverse and no duplicates.
    """
    def __init__(self)->None:
        self.catalog = {
            "Laptop": {
                "Gaming": {
                    "brands": ["ASUS", "MSI", "Lenovo", "Alienware"],
                    "features": [
                        "AI-enhanced performance",
                        "Advanced thermal cooling",
                        "High refresh-rate display",
                        "Premium build quality",
                        "Designed for AAA gaming",
                        "Fast SSD storage"
                    ],
                    "attributes": {
                        "Processor": ["Intel Core Ultra 9", "AMD Ryzen 9"],
                        "Graphics": ["RTX 4070", "RTX 4080"],
                        "Memory": ["32GB", "64GB"],
                        "Storage": ["1TB SSD", "2TB SSD"],
                        "Battery": ["80Wh", "90Wh"]
                    }
                },
                "Business": {
                    "brands": ["Dell", "HP", "Lenovo"],
                    "features": [
                        "Lightweight design",
                        "Long battery life",
                        "Enterprise-grade security",
                        "Fast multitasking",
                        "Ideal for professionals",
                        "Reliable everyday performance"
                    ],
                    "attributes": {
                        "Processor": ["Intel Core Ultra 7", "AMD Ryzen 7"],
                        "Graphics": ["Intel Arc", "Intel Iris Xe"],
                        "Memory": ["16GB", "32GB"],
                        "Storage": ["512GB SSD", "1TB SSD"],
                        "Battery": ["70Wh", "80Wh"]
                    }
                },
                "Budget": {
                    "brands": ["Acer", "ASUS", "HP"],
                    "features": [
                        "Affordable pricing",
                        "Energy efficient",
                        "Compact design",
                        "Reliable everyday computing",
                        "Student friendly"
                    ],
                    "attributes": {
                        "Processor": ["Intel Core i3", "AMD Ryzen 5"],
                        "Graphics": ["Intel UHD"],
                        "Memory": ["8GB", "16GB"],
                        "Storage": ["256GB SSD", "512GB SSD"],
                        "Battery": ["50Wh", "60Wh"]
                    }
                }
            },

            "Smartphone": {
                "Flagship": {
                    "brands": ["Samsung", "Apple", "Google"],
                    "features": [
                        "Professional-grade camera system",
                        "Ultra-smooth AMOLED display",
                        "Fast wireless charging",
                        "5G connectivity",
                        "AI-powered photography",
                        "Premium flagship experience"
                    ],
                    "attributes": {
                        "Chipset": ["Snapdragon 8 Elite", "Apple A18 Pro", "Tensor G5"],
                        "Display": ["6.7-inch AMOLED"],
                        "Storage": ["256GB", "512GB"],
                        "Battery": ["5000mAh"],
                        "Camera": ["50MP Triple Camera"]
                    }
                },
                "Budget": {
                    "brands": ["Redmi", "POCO", "Realme"],
                    "features": [
                        "Excellent value for money",
                        "Long battery life",
                        "Smooth everyday performance",
                        "Modern design",
                        "Fast charging support"
                    ],
                    "attributes": {
                        "Chipset": ["Snapdragon 6 Gen 1", "Helio G99"],
                        "Display": ["6.5-inch LCD"],
                        "Storage": ["128GB"],
                        "Battery": ["5000mAh"],
                        "Camera": ["50MP Dual Camera"]
                    }
                }
            },

            "Tablet": {
                "Premium": {
                    "brands": ["Apple", "Samsung"],
                    "features": [
                        "Large immersive display",
                        "Perfect for creativity",
                        "Excellent multimedia experience",
                        "Powerful multitasking"
                    ],
                    "attributes": {
                        "Chipset": ["Apple M4", "Snapdragon X Elite"],
                        "Display": ["12.9-inch OLED"],
                        "Storage": ["256GB", "512GB"],
                        "Battery": ["9000mAh"]
                    }
                }
            },

            "Monitor": {
                "Gaming": {
                    "brands": ["LG", "ASUS", "MSI"],
                    "features": [
                        "Ultra-smooth gameplay",
                        "High refresh-rate display",
                        "Low response time",
                        "Immersive viewing experience"
                    ],
                    "attributes": {
                        "Resolution": ["1440p", "4K"],
                        "Refresh Rate": ["165Hz", "240Hz"],
                        "Panel": ["IPS", "OLED"],
                        "Size": ["27-inch", "32-inch"]
                    }
                }
            },

            "Keyboard": {
                "Mechanical": {
                    "brands": ["Keychron", "Corsair", "Logitech"],
                    "features": [
                        "Tactile typing experience",
                        "Customizable RGB lighting",
                        "Durable mechanical switches",
                        "Comfortable for long sessions"
                    ],
                    "attributes": {
                        "Switch Type": ["Red", "Brown", "Blue"],
                        "Layout": ["TKL", "Full Size", "75%"],
                        "Connectivity": ["USB-C", "Bluetooth"],
                        "Backlight": ["RGB", "White"]
                    }
                }
            },

            "Mouse": {
                "Wireless": {
                    "brands": ["Logitech", "Razer", "SteelSeries"],
                    "features": [
                        "Precision tracking",
                        "Ergonomic design",
                        "Low latency wireless connection",
                        "Long battery life"
                    ],
                    "attributes": {
                        "Sensor": ["Optical", "Laser"],
                        "DPI": ["16000", "26000"],
                        "Connectivity": ["2.4GHz", "Bluetooth"]
                    }
                }
            },

            "Headphones": {
                "Wireless": {
                    "brands": ["Sony", "Bose", "Sennheiser"],
                    "features": [
                        "Immersive audio",
                        "Active noise cancellation",
                        "Comfortable all-day wear",
                        "Crystal-clear voice calls"
                    ],
                    "attributes": {
                        "Driver Size": ["40mm", "45mm"],
                        "Noise Cancellation": ["Yes"],
                        "Battery Life": ["30 Hours", "40 Hours"]
                    }
                }
            },

            "Smartwatch": {
                "Fitness": {
                    "brands": ["Apple", "Samsung", "Garmin"],
                    "features": [
                        "Comprehensive health tracking",
                        "Built-in GPS",
                        "Water resistant",
                        "Long-lasting battery"
                    ],
                    "attributes": {
                        "Display": ["AMOLED"],
                        "Battery Life": ["2 Days", "5 Days"],
                        "Water Resistance": ["5 ATM"],
                        "Sensors": ["Heart Rate", "SpO2", "GPS"]
                    }
                }
            }
        }

        self.used_signatures: set[tuple] = set()
        self.current_id: int = 1

    

    def generate_document(self) -> Document:
        """
        Generates a single realistic product document.
        """

       # Select category and profile
        category = random.choice(list(self.catalog.keys()))
        profile = random.choice(list(self.catalog[category].keys()))

        pool = self.catalog[category][profile]

        brand = random.choice(pool["brands"])

        # Generate specifications dynamically
        specifications = {}

        for attribute, values in pool["attributes"].items():
            specifications[attribute] = random.choice(values)

        # Duplicate signature
        signature = (
            category,
            profile,
            brand,
            tuple(sorted(specifications.items()))
        )

        if signature in self.used_signatures:
            return self.generate_document()

        self.used_signatures.add(signature)

        # Product name
        name = f"{brand} {profile} {category}"

        # Features 
        num_features = min(4, len(pool["features"]))

        features = random.sample(
            pool["features"],
            k=num_features
        )

        # Generic description
        description = (
            f"The {name} is a {profile.lower()} {category.lower()} "
            f"designed for users seeking {', '.join(features[:-1])}, "
            f"and {features[-1]}. "
            f"It delivers reliable performance and a premium user experience."
        )
        document = Document(
            document_id=self.current_id,
            name=name,
            category=category,
            features=features,
            specifications=specifications,
            description=description
        )

        self.current_id += 1

        return document
    def generate_dataset(self,num_documents: int) ->list[Document]:
        """
        Generate a synthetic dataset containing the specified number of 
        product documents.
        """
        dataset = []
        for i in range(num_documents):
            dataset.append(self.generate_document())

        return dataset

In [12]:
generator = SyntheticDatasetGenerator()

dataset = generator.generate_dataset(3)

for doc in dataset:
    print(doc)
    print("-" * 80)

Document(document_id=1, name='Logitech Mechanical Keyboard', category='Keyboard', features=['Customizable RGB lighting', 'Durable mechanical switches', 'Tactile typing experience', 'Comfortable for long sessions'], specifications={'Switch Type': 'Blue', 'Layout': '75%', 'Connectivity': 'Bluetooth', 'Backlight': 'RGB'}, description='The Logitech Mechanical Keyboard is a mechanical keyboard designed for users seeking Customizable RGB lighting, Durable mechanical switches, Tactile typing experience, and Comfortable for long sessions. It delivers reliable performance and a premium user experience.')
--------------------------------------------------------------------------------
Document(document_id=2, name='Apple Premium Tablet', category='Tablet', features=['Perfect for creativity', 'Large immersive display', 'Powerful multitasking', 'Excellent multimedia experience'], specifications={'Chipset': 'Apple M4', 'Display': '12.9-inch OLED', 'Storage': '256GB', 'Battery': '9000mAh'}, descripti

## EMBEDDING MODULE

### Base Embedder

In [13]:
import numpy as np
class BaseEmbedder(ABC):
    """
    This is a abstract class for all embedding models.

    Every embedder must convert a list of strings 
    to numerical vector embeddings.
    """
    @abstractmethod
    def embed(self,texts : list[str]) -> np.ndarray:
        """
        converts a batch of text strings into embeddings.
        
        Argumnets: texts - List of input text strings

        Returns: A NumPy array (number of texts, embedding_dimension).
        """
        pass


### Sentence Transfromer Embedder

In [14]:
from sentence_transformers import SentenceTransformer

In [15]:
class SentenceTransformerEmbedder(BaseEmbedder):
  def __init__(self,model_name: str ="all-MiniLM-L6-v2") ->None:
    self.model = SentenceTransformer(model_name)
  def embed(self, texts: list[str])->np.ndarray:
    if not texts:
      raise ValueError("Input text list cannot be empty.")
    if any(not text.strip() for text in texts):
      raise ValueError("Input contains empty or white-space text only.")
    return self.model.encode(texts,convert_to_numpy= True )

In [16]:
embedder = SentenceTransformerEmbedder()
texts = [
    "Gaming laptop with RTX 4080",
    "Business laptop with long battery life",
    "Wireless mechanical keyboard"
]
embeddings = embedder.embed(texts)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2804.15it/s]


In [17]:
print(embeddings)
print(embeddings.shape)
print(embeddings[0][0:100])

[[ 0.02724098  0.01770753  0.04533298 ...  0.01595695 -0.07210518
  -0.02229023]
 [ 0.01812536  0.11814071 -0.00189719 ... -0.08847365 -0.02093119
   0.03051163]
 [-0.05634657 -0.02059934  0.01858184 ... -0.0124944   0.03413037
  -0.00751804]]
(3, 384)
[ 0.02724098  0.01770753  0.04533298 -0.01194549 -0.05957933  0.04560719
  0.02771936  0.02996795 -0.06193102  0.04245458  0.01796274  0.03071201
  0.03759332  0.05193875  0.04804965  0.03755218  0.04584726 -0.08173543
 -0.00784013 -0.04936831 -0.01093941 -0.07882236 -0.05493359 -0.08288512
  0.01116535  0.08234805  0.10343518  0.05946497 -0.00890028  0.01693121
 -0.04074173 -0.03920551 -0.07804511  0.0202396  -0.08020975 -0.07650096
 -0.04059168 -0.02815543  0.01241434 -0.05786476 -0.09857851 -0.03661565
 -0.03599507  0.02732523  0.00356351 -0.05866133  0.01380735  0.00224144
  0.01145708  0.05051312 -0.00855291 -0.00884811 -0.00268463  0.01035629
 -0.04640779  0.07569     0.03038832 -0.00569937 -0.01143454 -0.08116189
  0.0295551  -0.1

## Embedding Record

In [18]:
@dataclass
class EmbeddingRecord:
    """
    stores a chunk and its corresponding embedding vector.
    """
    chunk_id: str
    embedding: np.ndarray
    metadata: dict

## Chunk Repository

Stores and manages Chunk objects independently of the vector store.

The repository allows chunks to be added, retrieved, deleted, and listed using
their unique chunk IDs.

In [19]:
class ChunkRepository:
    """
    Repository for storing and managing the chunk objects
    """
    def __init__(self)->None:
        self._chunks: dict[str,Chunk] = {}
    def add(self, chunk:Chunk)->None:
        """
        Adds a Chunk to the repo.
        Args: Chunk object to be stored.
        Raises Value error if ID already exists.
        """
        if chunk.chunk_id in self._chunks:
            raise ValueError(f"Chunk with ID '{chunk.chunk_id}' already exists.")
        self._chunks[chunk.chunk_id]= chunk
    def get(self,chunk_id: str) ->Chunk:
        """
        Retrieves a chunk by its unique ID.
        Args: Chunk_id - The unique ID of the chunk.
        Raises Key error if the chunk_id not found 
        """
        if chunk_id not in self._chunks:
            raise KeyError(f"Chunk with ID '{chunk_id} not found.'")
        return self._chunks[chunk_id]
    def delete(self,chunk_id: str) ->None:
        """
        Deletes a chunk from the repo.
        Args: chunk_id - Thee unique ID of chunk.
        Raises Key error if the chunk_id not found 
        """
        if chunk_id not in self._chunks:
            raise KeyError(f"Chunk with ID'{chunk_id}' not found.")
        del self._chunks[chunk_id]
    def get_all(self)->list[Chunk]:
        """
        Returns all the stored chunks (list of all the chunk objects).
        """
        return list(self._chunks.values())
    

#### Testcase (Check)

In [20]:
repository = ChunkRepository()

chunk1 = Chunk(
    chunk_id="chunk_1",
    document_id=1,
    text="Gaming laptop with RTX 4080",
    metadata={"category": "Laptop"},
    chunk_index=0
)

chunk2 = Chunk(
    chunk_id="chunk_2",
    document_id=2,
    text="Wireless mechanical keyboard",
    metadata={"category": "Keyboard"},
    chunk_index=0
)

repository.add(chunk1)
repository.add(chunk2)

print(repository.get("chunk_1"))

print("-" * 60)

for chunk in repository.get_all():
    print(chunk)

print("-" * 60)

repository.delete("chunk_1")

for chunk in repository.get_all():
    print(chunk)

Chunk(chunk_id='chunk_1', document_id=1, text='Gaming laptop with RTX 4080', metadata={'category': 'Laptop'}, chunk_index=0)
------------------------------------------------------------
Chunk(chunk_id='chunk_1', document_id=1, text='Gaming laptop with RTX 4080', metadata={'category': 'Laptop'}, chunk_index=0)
Chunk(chunk_id='chunk_2', document_id=2, text='Wireless mechanical keyboard', metadata={'category': 'Keyboard'}, chunk_index=0)
------------------------------------------------------------
Chunk(chunk_id='chunk_2', document_id=2, text='Wireless mechanical keyboard', metadata={'category': 'Keyboard'}, chunk_index=0)


## Vector Store Module
The vector store indexes the embedding vector and performs similarity search.

The main pro here is that it allows replacable interface that allows differnt storage backends (e.g ChromaDB,FAISS etc.) without ever changing the retreival pipeline.

In [21]:
class BaseVectorStore(ABC):
    """
    Abstract base class for all vector store implementations.

    Every vector store must support indexing, similarity search,
    deletion and metadata-based filtering.
    """
    @abstractmethod
    def add(self,record: EmbeddingRecord) ->None:
        """
        Adds a single embedding record to the vector store.

        Args: record - the embedding record to be indexed.
        """
        pass

    @abstractmethod
    def add_all(self,records: list[EmbeddingRecord])->None:
        """
        Adds multiple embedding records to the vector store.

        Args: records - list of embedding records to be indexed.
        """
        pass
    @abstractmethod
    def search(self,query_embedding: np.ndarray,top_k: int = 5,
    filters: dict | None = None)->list[tuple[str,float]]:
        """
        Searches for the most similar embedding vectors.
        Args:
              query_embedding - embedding of the input query.
              top_k - number of most similar results to return (default:5)
              filters - Optional metadata filters.
        """
        pass
    @abstractmethod
    def delete(self, chunk_id: str)->None:
        """
        Deletes an embedding record from the vector store.
        Args: chunk_id - unique id of the chunk.
        """
        pass
    @abstractmethod
    def clear(self) ->None:
        """
        Remove all embedding records from vector store.
        """
        pass

## In-Memory Vector Store
A simple in-memory implementation of the BaseVetorStore

It stores embedding records in a dictionary and perform a brute-force 
similarity search over all the internal embeddings.

In [22]:
from sklearn.metrics.pairwise import cosine_similarity
class InMemoryVectorStore(BaseVectorStore):
    """
    Stores embedding records in a dictionary keyed by chunk ID.
    """
    def __init__(self)->None:
        self._records : dict[str, EmbeddingRecord] = {} 
    def add(self,record :EmbeddingRecord) -> None:
        """
        Adds a single embedding record to the vector store.

        Args: record - the embedding record to be indexed.

        Raises ValueError if chunk id already exists.
        """
        if record.chunk_id in self._records:
            raise ValueError(f"Chunk with ID '{record.chunk_id}' already exists.")
        self._records[record.chunk_id] = record
    def add_all(self, records:EmbeddingRecord)->None:
        """
        Adds multiple embedding records to the vector store.

        Args: List of embedding records to be indexed.
        """
        for record in records:
            self.add(record)
    def search(self, query_embedding, top_k = 5,
                filters = None)->list[tuple[str,float]]:
        """
        Searches for the most similar embedding vectors.

        Args:
              query_embedding - embedding of the input query.
              top_k - number of most similar results to return (default:5)
              filters - Optional metadata filters.

        Returns a list of (chunk_id,similarity_score) tuples.
        """
        if top_k <=0:
            raise ValueError("top_k must be greater then 0.")
        if query_embedding.size == 0:
            raise ValueError("Query embedding cannot be empty.")
        results: list[tuple[str,float]] = []
        for record in self._records.values():
            if filters:
                if any(record.metadata.get(key)!= value
                for key,value in filters.items()):
                    continue
            similarity = cosine_similarity(query_embedding.reshape(1,-1),
                                                  record.embedding.reshape(1,-1)
                                                  )[0][0]
            results.append((record.chunk_id,float(similarity)))
        results.sort(key = lambda result: result[1],reverse=True)
        return results[:top_k]

        
    def delete(self,chunk_id :str)->None:
        """
        Deletes an embedding record from the vector store.

        Args: chunk_id: unique id of the chunk.

        Raises KeyError  if the chunk ID not exists.
        """
        if chunk_id not in self._records:
            raise KeyError(f"Chunk ID '{chunk_id}' not found.")
        del self._records[chunk_id]
    def clear(self)->None:
        """
        Wipes out all embedding from the vector store.
        """
        self._records.clear()

### Testing

In [23]:
vector_store = InMemoryVectorStore()

In [24]:
texts = [
    "Gaming laptop with RTX 4080",
    "Wireless mechanical keyboard",
    "Business laptop with long battery life"
]

embeddings = embedder.embed(texts)

records = [
    EmbeddingRecord(
        chunk_id="chunk_1",
        embedding=embeddings[0],
        metadata={"category": "Laptop"}
    ),
    EmbeddingRecord(
        chunk_id="chunk_2",
        embedding=embeddings[1],
        metadata={"category": "Keyboard"}
    ),
    EmbeddingRecord(
        chunk_id="chunk_3",
        embedding=embeddings[2],
        metadata={"category": "Laptop"}
    ),
]

In [25]:
vector_store.add_all(records)

In [26]:
query = "Gaming laptop"
query_embedding = embedder.embed([query])[0]
results = vector_store.search(
    query_embedding=query_embedding,
    top_k=2
)
print(results)

[('chunk_1', 0.6072567105293274), ('chunk_3', 0.4279588460922241)]


In [27]:
results = vector_store.search(
    query_embedding=query_embedding,
    top_k=2,
    filters={"category": "Keyboard"}
)
print(results)

[('chunk_2', 0.278918981552124)]


In [28]:
vector_store.delete("chunk_1")

results = vector_store.search(
    query_embedding=query_embedding,
    top_k=5
)
print(results)

[('chunk_3', 0.4279588460922241), ('chunk_2', 0.278918981552124)]


In [29]:
vector_store.clear()
results = vector_store.search(
    query_embedding=query_embedding,
    top_k=5
)
print(results)

[]


## Context Management Module

The context manager assembles the retrieved chunks ainto a final prompt that can be passed to a language model.

It is responsible for chubk selection, de-duplication, conetxt budgeting,
and prompt construction.

## Base ContextManager

In [30]:
class BaseContextManager:
    """
    Abstract base class for context management.

    Every context manager must assemble retreived chunks
    into a final prompt for the language model.
    """
    @abstractmethod
    def build_context(self,search_results: list[tuple[str,float]],
                      chunk_repository : ChunkRepository,
                      query: str
                      )->str:
        """
        Builds the final context prompt.
        Args: 
            search_results - list[tuple[str,float]],
            chunk_repository - repo. containing stored chunks.
            query - user's natural language query.

        Returns the final prompt string.
        """
        pass


In [31]:
class DefaultContextManager(BaseContextManager):
    """
    Default implementation of the context manager.

    Retreives chunks, removes duplicate, enforces a context budget,
    and constructs the final prompt.
    """
    def __init__(self, tokenizer_name: str = "sentence-transformers/all-MiniLM-L6-v2",
                 context_budget: int = 512)->None:
        """
        Initalizes the conetx manager.
        Args: 
              tokenizer_name - Hugging Face tokenizer used for token counting.
              conetext_budget - Maximum number of tokens allowed.
              in the received context.
        """
        if context_budget <=0:
            raise ValueError("Context_budget must be greater than 0.")
        self.context_budget = context_budget
        self.tokenizer = AutoTokenizer.from_pretrained(tokenizer_name)

    def _retrieve_chunks(self,search_results: list[tuple[str,float]],
                        chunk_repository: ChunkRepository
                        )-> list[Chunk]:
        """
        Retrieves Chunk objects from the repository.

        Args:
            search_results - List of (chunk_id, similarity_score) tuples.
            chunk_repository - Repository containing stored chunks.

        Returns a List of retrieved Chunk objects.
        """
        retrieved_chunks : list[Chunk] =[]
        for chunk_id, _ in search_results:
            retrieved_chunks.append(chunk_repository.get(chunk_id))
        return retrieved_chunks
    
    def _deduplicate_chunks(self,chunks: list[Chunk])->list[Chunk]:
        """
        Removes duplicate chunks while preserving order.

        Args:
            chunks - List of retrieved chunks.

        Returns aList of unique chunks.
        """
        unique_chunks: list[Chunk] = []
        seen_chunk_ids: set[str] =set()
        for chunk in chunks:
            if chunk.chunk_id not in seen_chunk_ids:
                unique_chunks.append(chunk)
                seen_chunk_ids.add(chunk.chunk_id)
        return unique_chunks
        
    def _apply_token_budget(self,chunks: list[Chunk]) ->str:
            
        """
            Applies the token budget and builds the context string.

            Args:
                chunks: List of unique chunks.

            Returns Context string within the configured token budget.
        """
            
        context =""
        current_tokens = 0

        for chunk in chunks:
            chunk_text = chunk.text
            token_ids = self.tokenizer.encode(
                            chunk_text,
                            add_special_tokens = False)
                
            chunk_tokens = len(token_ids)
            remaining_budget = self.context_budget - current_tokens
            if remaining_budget < 0:
                break
            if chunk_tokens <=remaining_budget:
                context +=chunk_text + "\n\n"
                current_tokens += chunk_tokens
            else:
                truncated_text = self.tokenizer.decode(
                    token_ids[:remaining_budget],
                    skip_special_tokens = True)
                context += truncated_text + "\n\n"
                break
        return context
    
    def _build_prompt(self,context: str,query : str) ->str:
            """
                Constructs the final prompt for the language model.

                Args:
                    context: Retrieved context.
                    query: User query.

                Returns:
                    Final prompt string.
            """
            
            return (
                "You are a helpful AI assistant.\n\n"
                "Use ONLY the provided context to answer the user's question.\n"
                "If the answer is not present in the context, say the information is unavilable.\n\n"
                "Context:\n"
                "--------------------------------------------------"
                f"{context}"
                "--------------------------------------------------"
                f"Question:{query}\n\n"
                "Answer:"
                )

        

    def build_context(self,search_results: list[tuple[str,float]],
                      chunk_repository : ChunkRepository,
                      query: str
                      )->str:
        """
        Builds the final context prompt for language model.
        Args: 
            search_results - list[tuple[str,float]],
            chunk_repository - repo. containing stored chunks.
            query - user's natural language query.

        Returns the final prompt string.
        """
        if not search_results:
            raise ValueError("Searchh resus cannot be empty.")
        if not query.strip():
            raise ValueError("Query cannot be empty.")
        chunks = self._retrieve_chunks(search_results,chunk_repository)

        chunks = self._deduplicate_chunks(chunks)
        
        context = self._apply_token_budget(chunks)

        return self._build_prompt(context,query)



## RAG Pipeline

Coordinates the complete retrieval pipeline by connecting the chunker,
embedder, repository, vector store, and context manager.

In [32]:
class RAGPipeline:
    """
    End-to-end retrieval pipeline.
    """
    def __init__(self,chunker: BaseChunker,
                 embedder: BaseEmbedder,
                 repository: ChunkRepository,
                 vector_store: BaseVectorStore,
                 context_manager: BaseContextManager) ->None:
        """
        Initialize RAG pipeline.
        """
        self.chunker = chunker
        self.embedder = embedder
        self.repository = repository
        self.vector_store = vector_store
        self.context_manager = context_manager

    def index_documents(self,documents: list[Document])->None:
        """
        Indexes a collection of documents into the RAG pipeline.
        Args: documents - list of docs. to be indexed.
        Raises ValueError if the document list is empty.
        """
        if not documents:
            raise ValueError("Document list cannot be empty.")
        
        all_chunks: list[Chunk] = []
        for document in documents:
            chunks = self.chunker.chunk(document)
            all_chunks.extend(chunks)
        for chunk in all_chunks:
            self.repository.add(chunk)
        chunk_texts = [chunk.text for chunk in all_chunks]
        embeddings = self.embedder.embed(chunk_texts)

        embedding_records: list[EmbeddingRecord] =[]
        for chunk, embedding in zip(all_chunks,embeddings):
            embedding_records.append(EmbeddingRecord(chunk_id = chunk.chunk_id,
                                    embedding = embedding,
                                    metadata = chunk.metadata))
        self.vector_store.add_all(embedding_records)
    def retrieve(self, query : str,top_k : int = 5, 
              filters: dict | None = None) ->str:
        """
        Retrieves the most relevant context for a user query.capitalize.

        Args:  
            query - user query.
            top_k - Number if chunks to retrieve.
            filters - Optional metadata filters.

        Returns final prompt constructed by the conetxt manager.

        Raises ValueError if the query is empty.
        """

        if not query.strip():
            raise ValueError("Query cannot be empty.")
        query_embedding = self.embedder.embed([query])[0]

        search_results = self.vector_store.search(query_embedding = query_embedding,
                                                top_k = top_k,
                                                filters = filters)
        prompt = self.context_manager.build_context(
            search_results = search_results,
            chunk_repository = self.repository,
            query = query
        )
        return prompt
    

## Final Testing

In [33]:
generator = SyntheticDatasetGenerator()

documents = generator.generate_dataset(500)

print(len(documents))

for document in documents[:2]:
    print(document)

500
Document(document_id=1, name='Samsung Flagship Smartphone', category='Smartphone', features=['Fast wireless charging', 'Premium flagship experience', 'Professional-grade camera system', 'Ultra-smooth AMOLED display'], specifications={'Chipset': 'Snapdragon 8 Elite', 'Display': '6.7-inch AMOLED', 'Storage': '256GB', 'Battery': '5000mAh', 'Camera': '50MP Triple Camera'}, description='The Samsung Flagship Smartphone is a flagship smartphone designed for users seeking Fast wireless charging, Premium flagship experience, Professional-grade camera system, and Ultra-smooth AMOLED display. It delivers reliable performance and a premium user experience.')
Document(document_id=2, name='Redmi Budget Smartphone', category='Smartphone', features=['Fast charging support', 'Modern design', 'Excellent value for money', 'Long battery life'], specifications={'Chipset': 'Helio G99', 'Display': '6.5-inch LCD', 'Storage': '128GB', 'Battery': '5000mAh', 'Camera': '50MP Dual Camera'}, description='The Re

In [34]:
chunker = FieldAwareChunker()

chunks = []

for document in documents:
    chunks.extend(chunker.chunk(document))

print(len(chunks))
print(chunks[:5])

2000
[Chunk(chunk_id='1_0', document_id=1, text='Product Name:Samsung Flagship Smartphone\nCategory: Smartphone', metadata={'category': 'Smartphone', 'section': 'identity'}, chunk_index=0), Chunk(chunk_id='1_1', document_id=1, text='- Fast wireless charging\n- Premium flagship experience\n- Professional-grade camera system\n- Ultra-smooth AMOLED display', metadata={'category': 'Smartphone', 'section': 'features'}, chunk_index=1), Chunk(chunk_id='1_2', document_id=1, text='Chipset:Snapdragon 8 Elite\nDisplay:6.7-inch AMOLED\nStorage:256GB\nBattery:5000mAh\nCamera:50MP Triple Camera', metadata={'category': 'Smartphone', 'section': 'specifications'}, chunk_index=2), Chunk(chunk_id='1_3', document_id=1, text='The Samsung Flagship Smartphone is a flagship smartphone designed for users seeking Fast wireless charging, Premium flagship experience, Professional-grade camera system, and Ultra-smooth AMOLED display. It delivers reliable performance and a premium user experience.', metadata={'cate

In [35]:
texts = [chunk.text for chunk in chunks]

embeddings = embedder.embed(texts)

print(len(embeddings))
print(embeddings[0].shape)

2000
(384,)


In [36]:
pipeline = RAGPipeline(
    chunker=FieldAwareChunker(),
    embedder=SentenceTransformerEmbedder(),
    repository=ChunkRepository(),
    vector_store=InMemoryVectorStore(),
    context_manager=DefaultContextManager()
)

pipeline.index_documents(documents)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1850.52it/s]


In [37]:
prompt = pipeline.retrieve(
    "Gaming laptop with RTX graphics"
)
print(prompt)

You are a helpful AI assistant.

Use ONLY the provided context to answer the user's question.
If the answer is not present in the context, say the information is unavilable.

Context:
--------------------------------------------------The ASUS Gaming Laptop is a gaming laptop designed for users seeking High refresh-rate display, Designed for AAA gaming, Premium build quality, and Advanced thermal cooling. It delivers reliable performance and a premium user experience.

The ASUS Gaming Laptop is a gaming laptop designed for users seeking High refresh-rate display, Fast SSD storage, AI-enhanced performance, and Advanced thermal cooling. It delivers reliable performance and a premium user experience.

The ASUS Gaming Laptop is a gaming laptop designed for users seeking Advanced thermal cooling, Premium build quality, High refresh-rate display, and AI-enhanced performance. It delivers reliable performance and a premium user experience.

The ASUS Gaming Laptop is a gaming laptop designed for

In [40]:
query = "Gaming laptop with RTX graphics"
query_embedding = pipeline.embedder.embed([query])[0]

results = pipeline.vector_store.search(
    query_embedding=query_embedding,
    top_k=100
)

for rank, (chunk_id, score) in enumerate(results, start=1):
    chunk = pipeline.repository.get(chunk_id)

    if (
        chunk.metadata["category"] == "Laptop"
        and chunk.metadata["section"] == "specifications"
    ):
        print(
            rank,
            score,
            chunk.text
        )

In [44]:
results = pipeline.vector_store.search(
    query_embedding=query_embedding,
    top_k=20
)

for rank, (chunk_id, score) in enumerate(results, start=1):
    chunk = pipeline.repository.get(chunk_id)

    print(rank, score)
    print(chunk.metadata)
    print("-" * 50)

1 0.5904688835144043
{'category': 'Laptop', 'section': 'description'}
--------------------------------------------------
2 0.5903456211090088
{'category': 'Laptop', 'section': 'description'}
--------------------------------------------------
3 0.5846493244171143
{'category': 'Laptop', 'section': 'description'}
--------------------------------------------------
4 0.5845367908477783
{'category': 'Laptop', 'section': 'description'}
--------------------------------------------------
5 0.5825791954994202
{'category': 'Laptop', 'section': 'description'}
--------------------------------------------------
6 0.5799621343612671
{'category': 'Laptop', 'section': 'description'}
--------------------------------------------------
7 0.5796173810958862
{'category': 'Laptop', 'section': 'description'}
--------------------------------------------------
8 0.5781656503677368
{'category': 'Laptop', 'section': 'description'}
--------------------------------------------------
9 0.5778009295463562
{'category'

In [45]:
count = 0

for chunk in pipeline.repository.get_all():
    if chunk.metadata["category"] == "Laptop":
        print(chunk.metadata)
        count += 1
        if count == 10:
            break

{'category': 'Laptop', 'section': 'identity'}
{'category': 'Laptop', 'section': 'features'}
{'category': 'Laptop', 'section': 'specifications'}
{'category': 'Laptop', 'section': 'description'}
{'category': 'Laptop', 'section': 'identity'}
{'category': 'Laptop', 'section': 'features'}
{'category': 'Laptop', 'section': 'specifications'}
{'category': 'Laptop', 'section': 'description'}
{'category': 'Laptop', 'section': 'identity'}
{'category': 'Laptop', 'section': 'features'}


In [46]:
query = "RTX 4080"

query_embedding = pipeline.embedder.embed([query])[0]

results = pipeline.vector_store.search(
    query_embedding=query_embedding,
    top_k=10
)

for chunk_id, score in results:
    chunk = pipeline.repository.get(chunk_id)
    print(score)
    print(chunk.metadata)
    print(chunk.text)
    print("=" * 70)

0.43905696272850037
{'category': 'Laptop', 'section': 'specifications'}
Processor:AMD Ryzen 9
Graphics:RTX 4080
Memory:64GB
Storage:1TB SSD
Battery:80Wh
0.43905696272850037
{'category': 'Laptop', 'section': 'specifications'}
Processor:AMD Ryzen 9
Graphics:RTX 4080
Memory:64GB
Storage:1TB SSD
Battery:80Wh
0.43905696272850037
{'category': 'Laptop', 'section': 'specifications'}
Processor:AMD Ryzen 9
Graphics:RTX 4080
Memory:64GB
Storage:1TB SSD
Battery:80Wh
0.43905696272850037
{'category': 'Laptop', 'section': 'specifications'}
Processor:AMD Ryzen 9
Graphics:RTX 4080
Memory:64GB
Storage:1TB SSD
Battery:80Wh
0.43826085329055786
{'category': 'Laptop', 'section': 'specifications'}
Processor:AMD Ryzen 9
Graphics:RTX 4080
Memory:32GB
Storage:1TB SSD
Battery:80Wh
0.43826085329055786
{'category': 'Laptop', 'section': 'specifications'}
Processor:AMD Ryzen 9
Graphics:RTX 4080
Memory:32GB
Storage:1TB SSD
Battery:80Wh
0.43826085329055786
{'category': 'Laptop', 'section': 'specifications'}
Processor: